# 📘 ComfyUI 起動ノートブック
### 本ノートブックは、書籍『**【キャラクターを創り動かす】画像・動画生成AI　スタートガイド**』のサポートコンテンツです。
### ComfyUI をGoogle Colabで起動し、AI画像生成を体験できます。
#### 💡Wan2.2 等のでの動画生成には GoogleColab の有料プランで使用できるランタイム（GPU）が必要になります。
#### 🌎Google Driveへの自動保存・Civitaiからのモデルダウンロードに対応しています。

---
## 🔑 事前準備：Civitai API キーの設定
モデルをダウンロードするには **Civitai の API キー** をGoogle Colabの「シークレット」に登録してください。

**① API キーの取得**
1. [https://civitai.com/user/account](https://civitai.com/user/account) を開く（要ログイン）
2. ページ下部の **API Keys** セクション → **＋ Add API key** をクリック
3. 名前を入力して **Save** → 表示されたキーをコピー

**② Google Colab シークレットへの登録**
1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
2. **＋ 新しいシークレットを追加** をクリック
3. 名前に `CIVITAI_KEY`、値にコピーしたAPIキーを貼り付けて保存
4. **ノートブックからのアクセス** をオンにする

> ⚠️ API キーはシークレットで管理し、コードに直接貼り付けないでください。

---
**ノートブック原作：** ざすこ（道草 雑草子）  
X: [@zasuko_michiksa](https://x.com/zasuko_michiksa) ／ YouTube: [ＡＩみちくさｃｈ](https://www.youtube.com/channel/UC84fyKjiilxssZVxhE_RiaA)


# 🥚各種モデルのDL ＆ ComfyUIの起動🐣
##＜はじめる前の準備＞※重要※
### ❶生成に必要な各種モデルファイルのDL用URLを格納する各フォルダの記入欄に記載して下さい。
└🌐GoogleColab上の 📂ComfyUI / models  以下に、生成に必要な各種ファイルをDLします。（DL容量が少ない程起動が速くなります。）
### ❷（▶）ボタンをクリックしてComfyUIを起動


In [ ]:
# @markdown # 👈この（▶）ボタンをクリックしてComfyUIを起動（☕起動まで約10数分）
# @markdown ### ※停止と再起動もこのボタンで行います。
# @markdown ## 🌱実行ログに青字で👀「🚀 Pinggy URL: http:……」が表示されたら🖱️クリックしてください。

# ==========================================
# 🔧 PyTorch系ライブラリの軽量チェック（再起動ループ回避版）
# ==========================================
# 重要：このセル内では、現在のColabカーネルに torch / triton を直接 import しません。
# 理由：torchをimportすると内部でtritonも読み込まれ、その後pipがtriton周辺を確認しただけで
#       Colabが「セッションを再起動してください」と表示しやすくなるためです。
# 方針：別プロセスで確認し、必要な場合だけ修復します。
import sys, subprocess

CORE_TORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def torch_stack_ok():
    check_code = r'''
import sys
try:
    import torch, torchvision, torchaudio
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("torchaudio:", torchaudio.__version__)
    print("torch cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    ok = ("+cu128" in torch.__version__) and ("+cu128" in torchvision.__version__) and ("+cu128" in torchaudio.__version__)
    sys.exit(0 if ok else 2)
except Exception as e:
    print("PyTorch check failed:", repr(e))
    sys.exit(1)
'''
    result = subprocess.run([sys.executable, "-c", check_code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    return result.returncode == 0

if not torch_stack_ok():
    print("=" * 70)
    print("🔧 torch / torchvision / torchaudio を CUDA 12.8版にそろえます")
    print("※必要な時だけ修復します。修復後に再起動表示が出た場合は、その時だけ再起動してください。")
    print("=" * 70)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *CORE_TORCH_PACKAGES,
         "--index-url", "https://download.pytorch.org/whl/cu128"])
    if not torch_stack_ok():
        raise RuntimeError("torch / torchvision / torchaudio の修復に失敗しました。ランタイムを新規にして再実行してください。")

print("✅ PyTorch系ライブラリ確認OK。このままComfyUI起動処理に進みます。")

# using_L4_GPU = True # @param {type:"boolean"}
using_L4_GPU = False
include_manager = True # @param {type:"boolean"}

# ==========================================
# 📁 Google Drive マウント
# ==========================================
# @markdown ## 📁 Google Driveに接続
# @markdown ComfyUIの出力ファイルをバックアップする場合はチェックを入れてください

# @markdown ---
# @markdown ### Google Driveを使用する
use_google_drive = True  # @param {type:"boolean"}

# @markdown ---

if use_google_drive:
    from google.colab import drive
    drive.mount('/content/drive')

    print("=" * 70)
    print("✅ Google Driveがマウントされました")
    print("📁 マウント先: /content/drive/MyDrive")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Driveは使用しません")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("   （ランタイム終了時に消えるので注意してください）")
    print("=" * 70)


# ==========================================
# 📤 Google Drive 出力設定
# ==========================================
# @markdown ## 📤 Google Drive 出力設定
# @markdown ### ComfyUIの出力先をGoogle Driveに変更する
enable_gdrive_output = True  # @param {type:"boolean"}

# @markdown ### 出力先フォルダ設定
GDRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_output"  # @param {type:"string"}

# @markdown ---

# 出力フォルダの作成
if use_google_drive and enable_gdrive_output:
    from pathlib import Path
    output_path = Path(GDRIVE_OUTPUT)

    if output_path.exists():
        print(f"📁 既存のフォルダを使用: {GDRIVE_OUTPUT}")
    else:
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"📁 新規フォルダを作成: {GDRIVE_OUTPUT}")

    print("=" * 70)
    print("✅ ComfyUIの出力先がGoogle Driveに設定されます")
    print(f"📁 保存先: {GDRIVE_OUTPUT}")
    print("💡 生成したファイルが直接Google Driveに保存されます")
    print("=" * 70)
elif enable_gdrive_output and not use_google_drive:
    print("=" * 70)
    print("⚠️ Google Drive出力が有効ですが、Google Driveがマウントされていません")
    print("💡 use_google_driveにチェックを入れてください")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Drive出力は無効です")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("=" * 70)


%cd /content
from IPython.display import clear_output
clear_output()
# torch / triton はColab標準または上のチェック結果を使います。
# xformersだけを --no-deps で入れ、torch/tritonを勝手に入れ替えないようにします。
!python -m pip install -q torchsde einops diffusers accelerate
!python -m pip install -q --no-deps xformers==0.0.32.post1
# pipがtorch/triton系を別バージョンへ動かさないための制約ファイル
# Colabの通常Pythonセルでは bash の here-document が壊れやすいため、Pythonで安全に書き出します。
with open('/content/torch_core_constraints.txt', 'w', encoding='utf-8') as f:
    f.write('''torch==2.8.0
torchvision==0.23.0
torchaudio==2.8.0
triton==3.4.0
xformers==0.0.32.post1
numpy==2.0.2
''')
!pip install av spandrel albumentations onnx opencv-python onnxruntime
!pip install color-matcher
!pip install --upgrade "protobuf>=5.29.1"
!pip install onnxruntime-gpu -y
!git clone https://github.com/comfyanonymous/ComfyUI
!pip install -r /content/ComfyUI/requirements.txt -c /content/torch_core_constraints.txt
clear_output()

%cd /content/ComfyUI/custom_nodes
if include_manager:
    !git clone https://github.com/ltdrdata/ComfyUI-Manager

# 既存のカスタムノード
#!git clone https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
#!git clone https://github.com/Isi-dev/ComfyUI_DeleteModelPassthrough.git
#!git clone https://github.com/Isi-dev/comfyui_controlnet_aux
#!git clone https://github.com/kijai/ComfyUI-WanVideoWrapper
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite
!git clone https://github.com/kijai/ComfyUI-KJNodes.git
#!git clone https://github.com/kijai/ComfyUI-segment-anything-2
#!git clone https://github.com/kijai/ComfyUI-Florence2
#!git clone https://github.com/john-mnz/ComfyUI-Inspyrenet-Rembg.git
#!git clone https://github.com/Isi-dev/ComfyUI_Animation_Nodes_and_Workflows

# 🆕 追加のカスタムノード
!git clone https://github.com/rgthree/rgthree-comfy.git
!git clone https://github.com/cubiq/ComfyUI_essentials.git
#!git clone https://github.com/chrisgoringe/cg-use-everywhere.git
!git clone https://github.com/aria1th/ComfyUI-LogicUtils.git
!git clone https://github.com/GACLove/ComfyUI-VFI.git
!git clone https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git
!git clone https://github.com/princepainter/ComfyUI-PainterI2V.git
# requirements.txtのインストール（既存）
#%cd /content/ComfyUI/custom_nodes/ComfyUI-Custom-Scripts
#!pip install -r requirements.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
#%cd /content/ComfyUI/custom_nodes/ComfyUI_DeleteModelPassthrough
#!pip install -r requirements.txt
#%cd /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
#!pip install -r requirements.txt
#%cd /content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper
#!pip install -r requirements.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI-KJNodes
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
#%cd /content/ComfyUI/custom_nodes/ComfyUI-Florence2
#!pip install -r requirements.txt
#%cd /content/ComfyUI/custom_nodes/ComfyUI-Inspyrenet-Rembg
#!pip install -r requirements.txt
#%cd /content/ComfyUI/custom_nodes/ComfyUI_Animation_Nodes_and_Workflows
#!pip install -r requirements.txt

# 🆕 追加ノードのrequirements.txt
%cd /content/ComfyUI/custom_nodes/rgthree-comfy
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI_essentials
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
#%cd /content/ComfyUI/custom_nodes/cg-use-everywhere
#!pip install -r requirements.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI-LogicUtils
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
%cd /content/ComfyUI/custom_nodes/ComfyUI-VFI
!pip install -r requirements.txt -c /content/torch_core_constraints.txt
%cd /content/ComfyUI/custom_nodes/comfyui-frame-interpolation
!pip install -r requirements.txt -c /content/torch_core_constraints.txt

if include_manager:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -r requirements.txt -c /content/torch_core_constraints.txt

# 🆕 protobufの最終調整
print("=" * 70)
print("🔧 Fixing protobuf version conflicts...")
!pip install protobuf==5.29.1 -c /content/torch_core_constraints.txt
print("✅ Protobuf version fixed")
print("=" * 70)

clear_output()

%cd /content/ComfyUI

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
from pathlib import Path

def install_apt_packages():
    packages = ['aria2']

    try:
        # Run apt install silently (using -qq)
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")

print("Installing apt packages...")
install_apt_packages()

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras"):
    import os
    from urllib.parse import urlparse, unquote

    # Hugging Faceなど、URLの末尾に正式なファイル名が入っている通常リンク用
    path_name = os.path.basename(urlparse(link).path)
    filename = unquote(path_name) if path_name else "downloaded_model.safetensors"
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \"{link}\" -d \"{folder}\" -o \"{filename}\""

    print("Executing download command:")
    print(command)

    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)

    return filename

def add_civitai_token_to_url(civitai_link, civitai_token=None):
    """CivitaiのURLにtokenを安全に追加する。既にtokenがある場合はそのまま使う。"""
    from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

    parsed = urlparse(civitai_link)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError("Invalid Civitai URL format. Please use a link like: https://civitai.com/api/download/models/1523247?...")

    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    if civitai_token and "token" not in query:
        query["token"] = civitai_token

    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, urlencode(query), parsed.fragment))

def pick_new_downloaded_file(folder, before_files):
    """wget実行後に増えたファイルから、実体のある最新ファイルを推定する。"""
    import os

    after_files = set(os.listdir(folder))
    candidates = []
    for name in after_files - before_files:
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            # wgetの一時ファイルやログっぽいものは除外
            if not name.endswith((".tmp", ".aria2")):
                candidates.append(path)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    """
    Civitai専用ダウンロード。
    重要：-Oで固定名を付けず、wgetの --content-disposition / --trust-server-names を使う。
    これにより、Civitaiが返す正式ファイル名（例：FlowCamera_epoch67.safetensors）で保存する。
    """
    import os
    import subprocess

    os.makedirs(folder, exist_ok=True)
    civitai_url = add_civitai_token_to_url(civitai_link, civitai_token)

    before_files = set(os.listdir(folder))

    cmd = [
        "wget",
        "--max-redirect=10",
        "--content-disposition",
        "--trust-server-names",
        "--show-progress",
        "-P", folder,
        civitai_url,
    ]

    print("Downloading from Civitai with official filename...")
    print("$ " + " ".join([f'\"{c}\"' if " " in c or "&" in c or "?" in c else c for c in cmd]))

    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print("❌ Civitai download failed.")
        return False

    downloaded_path = pick_new_downloaded_file(folder, before_files)

    # 同名ファイルが既に存在していて新規差分が取れない場合の保険：フォルダ内の最新safetensorsを表示
    if downloaded_path is None:
        safetensors = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith((".safetensors", ".ckpt", ".pt", ".pth", ".bin")) and os.path.isfile(os.path.join(folder, f))
        ]
        if safetensors:
            downloaded_path = max(safetensors, key=os.path.getmtime)

    if downloaded_path and os.path.exists(downloaded_path) and os.path.getsize(downloaded_path) > 0:
        filename = os.path.basename(downloaded_path)
        print(f"✅ Civitai model downloaded successfully: {downloaded_path}")
        print(f"📛 Saved filename: {filename}")
        return filename

    print("❌ Civitai download finished, but the saved file could not be detected.")
    return False

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    """
    Download a model file, automatically detecting if it's a Civitai link or huggingface download.

    Args:
        link: The download URL (either huggingface or Civitai)
        folder: Destination folder for the download
        civitai_token: Optional token for Civitai downloads (required if link is from Civitai)

    Returns:
        The filename of the downloaded model
    """
    if "civitai.com" in link.lower():
        if not civitai_token:
            print("⚠️ Civitaiトークンが未設定のため、トークンなしでダウンロードを試します。")
            print("   非公開・年齢制限・ログイン必須モデルの場合は失敗することがあります。")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True) -> bool:
    """
    Colab-optimized download with aria2c

    Args:
        url: Download URL
        dest_dir: Target directory (will be created if needed)
        filename: Optional output filename (defaults to URL filename)
        silent: If True, suppresses all output (except errors)

    Returns:
        bool: True if successful, False if failed
    """
    try:
        # Create destination directory
        Path(dest_dir).mkdir(parents=True, exist_ok=True)

        # Set filename if not specified
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]  # Remove URL parameters

        # Build command
        cmd = [
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '-d', dest_dir,
            '-o', filename,
            url
        ]

        # Add silent flags if requested
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)

        # Run download
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)

        if silent:
            print("Done!")
        else:
            print(f"Downloaded {filename} to {dest_dir}")
        return filename

    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"\nError downloading {filename}: {error}")
        return False
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# ========================================
# ダウンロード処理関数
# ========================================

def parse_urls(url_string):
    """
    複数行のURL文字列をパースして、有効なURLのリストを返す

    Args:
        url_string: 改行区切りのURL文字列

    Returns:
        list: 有効なURLのリスト
    """
    urls = []
    for line in url_string.strip().split('\n'):
        line = line.strip()
        # コメント行（#始まり）と空行をスキップ
        if line and not line.startswith('#') and line.startswith('http'):
            urls.append(line)
    return urls

def download_models_to_folder(url_string, folder_path, civitai_token=None):
    """
    指定されたフォルダに複数のモデルをダウンロード

    Args:
        url_string: 改行区切りのURL文字列
        folder_path: ダウンロード先フォルダパス
        civitai_token: Civitai APIトークン（オプション）
    """
    urls = parse_urls(url_string)

    if not urls:
        return

    print(f"\n📂 Downloading to: {folder_path}")
    print(f"   Files to download: {len(urls)}")

    for i, url in enumerate(urls, 1):
        if "civitai.com" in url.lower():
            filename = "Civitai official filename"
        else:
            filename = url.split('/')[-1].split('?')[0]
        print(f"   [{i}/{len(urls)}] {filename}...")

        # Civitai URLの場合はdownload_lora関数を使用
        if "civitai.com" in url.lower():
            download_lora(url, folder=folder_path, civitai_token=civitai_token)
        else:
            model_download(url, folder_path)

    print(f"✓ Completed: {folder_path}\n")


# ========================================
# 📥 各種モデルのダウンロード設定
# ========================================

# @markdown # 📥 モデルダウンロード設定
# @markdown 各フォルダごとに、使用したいモデルのダウンロードURLを入力してください。
# @markdown 複数のURLは改行で区切って入力できます。

# @markdown ---
# @markdown ## 🔑 Civitai API Token
# @markdown 👈のGoogle Colab「鍵のアイコン」 から `CIVITAI_KEY` という名前で登録したAPIキーを自動で読み込みます。

# @markdown 　└ スライドスイッチを✅にして、ノートブックからのアクセスを有効化して下さい。

civitai_token = ""

try:
    from google.colab import userdata
    # シークレットから取得を試みる
    civitai_token = userdata.get('CIVITAI_KEY')

    if civitai_token:
        civitai_token = civitai_token.strip()
        print("✅ Civitai API キーをシークレットから正常に読み込みました")
    else:
        print("⚠️ 'CIVITAI_KEY' は見つかりましたが、中身が空っぽのようです")

except Exception as e:
    print(f"❌ シークレットの読み込みに失敗しました: {e}")
    print("--------------------------------------------------")
    print("💡 次の2点を確認してください：")
    print("1. 左側の 🔑 (鍵アイコン) で、名前が『CIVITAI_KEY』(すべて大文字) になっていますか？")
    print("2. その横にある『ノートブックのアクセス権』のスイッチが【ON】になっていますか？")
    print("--------------------------------------------------")
    civitai_token = ""

# 最終確認
if not civitai_token:
    print("⚠️ 現在、キー設定なしの状態で動作しています。")

# @markdown ---
# @markdown ## 📁 diffusion_models フォルダ
# @markdown 各種メインモデル（動画生成など）
diffusion_models_url_1 = "https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf" # @param {type:"string"}
diffusion_models_url_2 = "https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf" # @param {type:"string"}
diffusion_models_url_3 = "" # @param {type:"string"}
diffusion_models_url_4 = "" # @param {type:"string"}
diffusion_models_url_5 = "" # @param {type:"string"}
diffusion_models_url_6 = "" # @param {type:"string"}
diffusion_models_url_7 = "" # @param {type:"string"}
diffusion_models_url_8 = "" # @param {type:"string"}

# 入力されたURLを結合
diffusion_models_urls = "\n".join([
    url for url in [
        diffusion_models_url_1,
        diffusion_models_url_2,
        diffusion_models_url_3,
        diffusion_models_url_4,
        diffusion_models_url_5,
        diffusion_models_url_6,
        diffusion_models_url_7,
        diffusion_models_url_8
    ] if url.strip()
])

# @markdown ---
# @markdown ## 📁 text_encoders フォルダ
# @markdown テキストエンコーダー（UMT5等）
text_encoders_url_1 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors" # @param {type:"string"}
text_encoders_url_2 = "" # @param {type:"string"}
text_encoders_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
text_encoders_urls = "\n".join([url for url in [text_encoders_url_1, text_encoders_url_2, text_encoders_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 vae フォルダ
# @markdown VAE（変分オートエンコーダー）
vae_url_1 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors" # @param {type:"string"}
vae_url_2 = "" # @param {type:"string"}
vae_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
vae_urls = "\n".join([url for url in [vae_url_1, vae_url_2, vae_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 clip_vision フォルダ
# @markdown CLIP Visionモデル（画像理解用）
clip_vision_url_1 = "" # @param {type:"string"}
clip_vision_url_2 = "" # @param {type:"string"}
clip_vision_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
clip_vision_urls = "\n".join([url for url in [clip_vision_url_1, clip_vision_url_2, clip_vision_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 loras フォルダ
# @markdown LoRAモデル（高速化、ライティング調整等）

loras_url_1 = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank256_bf16.safetensors" # @param {type:"string"}
loras_url_2 = "https://civitai.com/api/download/models/2155019?type=Model&format=SafeTensor" # @param {type:"string"}
loras_url_3 = "" # @param {type:"string"}
loras_url_4 = "" # @param {type:"string"}
loras_url_5 = "" # @param {type:"string"}
loras_url_6 = "" # @param {type:"string"}
loras_url_7 = "" # @param {type:"string"}
loras_url_8 = "" # @param {type:"string"}

# URLをまとめる
loras_urls = "\n".join([url for url in [loras_url_1, loras_url_2, loras_url_3, loras_url_4, loras_url_5, loras_url_6, loras_url_7, loras_url_8] if url])

# @markdown ---
# @markdown ## 📁 audio_encoders フォルダ
# @markdown Audio Encoderモデル（音声付き動画生成用）
audio_encoders_url_1 = "" # @param {type:"string"}
audio_encoders_url_2 = "" # @param {type:"string"}
audio_encoders_url_3 = "" # @param {type:"string"}

# URLをまとめる
audio_encoders_urls = "\n".join([url for url in [audio_encoders_url_1, audio_encoders_url_2, audio_encoders_url_3] if url])

# @markdown ---
# @markdown ## 📁 sam2 フォルダ
# @markdown SAM2セグメンテーションモデル
sam2_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 unet フォルダ
# @markdown UNetモデル（Flux等）
unet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 checkpoints フォルダ
# @markdown チェックポイントモデル
checkpoints_url_1 = "" # @param {type:"string"}
checkpoints_url_2 = "" # @param {type:"string"}
checkpoints_url_3 = "" # @param {type:"string"}
checkpoints_url_4 = "" # @param {type:"string"}
checkpoints_url_5 = "" # @param {type:"string"}

# 入力されたURLを結合
checkpoints_urls = "\n".join([url for url in [checkpoints_url_1, checkpoints_url_2, checkpoints_url_3, checkpoints_url_4, checkpoints_url_5] if url.strip()])

# @markdown ---
# @markdown ## 📁 controlnet フォルダ
# @markdown ControlNetモデル
controlnet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 clip フォルダ
# @markdown CLIPモデル
clip_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 upscale_models フォルダ
# @markdown アップスケールモデル
upscale_models_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 embeddings フォルダ
# @markdown Embeddingsモデル
embeddings_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 カスタムフォルダ（自由指定）
# @markdown フォルダパスとURLを指定
custom_folder_path = "" # @param {type:"string"}
custom_folder_urls = "" # @param {type:"string"}


# ========================================
# ダウンロード実行
# ========================================

print("=" * 70)
print("🚀 Starting model downloads...")
print("=" * 70)

# 各フォルダへのダウンロード
download_models_to_folder(diffusion_models_urls, "/content/ComfyUI/models/diffusion_models", civitai_token)
download_models_to_folder(text_encoders_urls, "/content/ComfyUI/models/text_encoders", civitai_token)
download_models_to_folder(vae_urls, "/content/ComfyUI/models/vae", civitai_token)
download_models_to_folder(clip_vision_urls, "/content/ComfyUI/models/clip_vision", civitai_token)
download_models_to_folder(loras_urls, "/content/ComfyUI/models/loras", civitai_token)
download_models_to_folder(audio_encoders_urls, "/content/ComfyUI/models/audio_encoders", civitai_token)
download_models_to_folder(sam2_urls, "/content/ComfyUI/models/sam2", civitai_token)
download_models_to_folder(unet_urls, "/content/ComfyUI/models/unet", civitai_token)
download_models_to_folder(checkpoints_urls, "/content/ComfyUI/models/checkpoints", civitai_token)
download_models_to_folder(controlnet_urls, "/content/ComfyUI/models/controlnet", civitai_token)
download_models_to_folder(clip_urls, "/content/ComfyUI/models/clip", civitai_token)
download_models_to_folder(upscale_models_urls, "/content/ComfyUI/models/upscale_models", civitai_token)
download_models_to_folder(embeddings_urls, "/content/ComfyUI/models/embeddings", civitai_token)

# カスタムフォルダ
if custom_folder_path and custom_folder_urls.strip():
    download_models_to_folder(custom_folder_urls, custom_folder_path, civitai_token)

print("=" * 70)
print("✓ All downloads completed!")
print("=" * 70)

# ========================================
# 🆕 サンプル画像の自動ダウンロード（inputフォルダへ格納）
# ========================================
print("\n📥 Downloading sample image to input folder...")
input_dir = "/content/ComfyUI/input"

# 既存のサンプル画像
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_i2v_image_SAMPLE_mei.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_i2v_image_SAMPLE_rhyth.png", input_dir)

# 追加のサンプル画像（全6点）
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_mei_01.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_mei_02.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_mei_chibi.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_rhyth_chibi.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_mei_logo_01.png", input_dir)
model_download("https://raw.githubusercontent.com/aicuai/Book-SG26/main/WebUI_Launch_Setup_Files/sg26_FLF2v_image_SAMPLE_mei_logo_02.png", input_dir)

print("✓ All sample images downloaded!\n")



# ========================================
# ComfyUI起動設定
# ========================================

# 🆕 出力ディレクトリの設定
output_dir_arg = ""
if use_google_drive and enable_gdrive_output:
    output_dir_arg = f"--output-directory {GDRIVE_OUTPUT}"
    print("=" * 70)
    print(f"✅ ComfyUI output will be saved directly to Google Drive")
    print(f"📁 Output path: {GDRIVE_OUTPUT}")
    print("=" * 70)

print("🚀 Pinggy Tunnel を準備しています...")
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI finished loading... setting up Pinggy tunnel!\n")

    p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-p", "443", "-R0:localhost:{}".format(port), "a.pinggy.io"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stdout:
        l = line.decode()
        if ("http://" in l or "https://" in l) and "dashboard.pinggy.io" not in l:
            print("\n" + "="*70)
            print("🚀 Pinggy URL:", l.strip())
            print("="*70 + "\n")
            print("✅URLからリンクを開いたら Pinggy（60分無料枠）の赤色の🟥「Enter Site」のボタンをクリックして下さい。")
            print("60分を超過して切断されたら「▶️実行ボタン」をクリックして処理を一旦停止し、その後再度クリックして ComfyUI を再起動させてください。")
            break

clear_output()
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ★ 最重要修正箇所: --listen 0.0.0.0 と --enable-cors-header を追加して 403 Forbidden を回避します
if using_L4_GPU:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header --cache-none --dont-print-server".strip()
    get_ipython().system(command)
else:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header --dont-print-server".strip()
    get_ipython().system(command)



# 🌱📋各種モデル_DL元リポジトリのメモ🌱
###　下記リンクのリポジトリから必要なモデルファイルのダウンロードURLを取得してきてください。
---
#🎬Wan2.2用 Diffusion Model
##＜ti2v_5B_fp16＞ ※標準テンプレートに対応
####※下記リンクをそのまま使用可能
- wan2.2_ti2v_5B_fp16：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors)

##＜t2v_14B_fp8＞ ※標準テンプレートに対応
####※下記リンクをそのまま使用可能
- Wan2.2-T2V-A14B(High_noise)：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_t2v_high_noise_14B_fp8_scaled.safetensors)
- Wan2.2-T2V-A14B(Low_noise)：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_t2v_low_noise_14B_fp8_scaled.safetensors)
##＜i2v_14B_fp8＞ ※標準テンプレートに対応
####※下記リンクをそのまま使用可能
- Wan2.2-I2V-A14B(High_noise)：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors)
- Wan2.2-I2V-A14B(Low_noise)：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors)

##＜t2v_GGUF＞ ※ModelLoaderの変更が必要
####※下記リンクをそのまま使用可能（GGUF_Q8）
- Wan2.2-T2V-A14B-HighNoise-Q8_0.gguf：(https://huggingface.co/QuantStack/Wan2.2-T2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-T2V-A14B-HighNoise-Q8_0.gguf)
- Wan2.2-T2V-A14B-LowNoise-Q8_0.gguf：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf)

####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- Wan2.2-T2V-A14B-GGUF：https://huggingface.co/QuantStack/Wan2.2-T2V-A14B-GGUF/tree/main
##＜i2v_GGUF＞ ※ModelLoaderの変更が必要
####※下記リンクをそのまま使用可能（GGUF_Q8）
- Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q8_0.gguf)
- Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q8_0.gguf)

- Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf)
- Wan2.2-T2V-A14B-LowNoise-Q4_K_M.gguf：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf)

####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- Wan2.2-I2V-A14B-GGUF：(https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/tree/main)

##＜Fun&Inp＞ ※動画の人物の動きを再現
####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- Wan2.2-Fun＆Inpl：(https://huggingface.co/Kijai/WanVideo_comfy_fp8_scaled/tree/main/Fun)

##＜Animate_GGUF＞ ※動画中の被写体をすげ替えor参照画像に反映
####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- Wan2.2-Animate-A14B-GGUF(https://huggingface.co/QuantStack/Wan2.2-Animate-14B-GGUF/tree/main)

#🎬Wan2.2用LoRA
##＜lightx2v_4steps（高速化）＞※High＆Low 別々に指定
####※下記リンクをそのまま使用可能
- LightX2v_t2v-4step_High：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_high_noise.safetensors)
- LightX2v_t2v-4step_Low：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_t2v_lightx2v_4steps_lora_v1.1_low_noise.safetensors)
- LightX2v_i2v-4step_High：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors)
- LightX2v_i2v-4step_Low：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors)
##＜lightx2v_Rank（高速化）＞※High＆Low共通
####※下記リンクをそのまま使用可能
- lightx2v_I2V_14B_480p_cfg_step_distill_rank256_bf16.safetensors：(https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank256_bf16.safetensors?download=true)
- lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank256_bf16.safetensors
：(https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lightx2v/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank256_bf16.safetensors?download=true)

- LightX2v_t2v&i2v-Rank：(https://huggingface.co/Kijai/WanVideo_comfy/tree/main/Lightx2v)

##＜relight_LoRA（Animate用）＞ ※Wan2.2 Animateに必須のLoRA
####※下記リンクをそのまま使用可能
- Wan2.2_relight：(https://huggingface.co/Kijai/WanVideo_comfy/tree/main/LoRAs/Wan22_relight)

#🎬Wan2.2用VAE
##＜VAE＞
####※下記リンクをそのまま使用可能
- Wan2.1_VAE.safetensors：(https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors)
- wan2.2_vae.safetensors：(https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors)

#🎬Wan2.2用TextEncoder
##＜TextEncoder＞
####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- umt5_xxl_fp16.safetensors：
(https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/tree/main/split_files/text_encoders)

#🎬Wan2.2用ClipVision
##＜ClipVision＞
####※下記リポジトリ内のモデルファイルのダウンロードURLを任意に取得
- clip_vision_h.safetensors：(https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/tree/main/split_files/clip_vision)

#🖼 StableDiffsions XL
##＜Checkpoint＞
Sierunami.v1：(https://civitai.com/api/download/models/1176276)

mellow_pencil-XL-v1.0.0-base：(https://huggingface.co/bluepen5805/mellow_pencil-XL/resolve/main/mellow_pencil-XL-v1.0.0-base.safetensors)

mellow_pencil-XL-v1.0.0-chaos：(https://huggingface.co/bluepen5805/mellow_pencil-XL/resolve/main/mellow_pencil-XL-v1.0.0-chaos.safetensors)

##＜VAE＞
####※下記リンクをそのまま使用可能
sdxl_vae：(https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors)

---
